# 09 — Run the application

Execute and relocate a complete scientific result, then explore the Kubernetes runner from the same notebook.

Run this lab in the **dev-platform** workspace on Minikube or Kubernetes, using the `eoepca/pde-code-server:amd64` image. Select its preinstalled **Bash** kernel and use **Shift+Enter**, reading each explanation before executing its cell. No terminal or kernel installation is needed. Stop at an unexpected error and fix it before continuing. The course environment is provisioned by workspace startup.

The kernel keeps environment variables and the working directory between cells. The first cell finds the course checkout and activates its locked environment. Command groups run in a subshell with `set -euo pipefail` so a failing command stops that cell without killing the kernel. Scratch results stay in `.notebook-runs/09-run-the-application/`; rerunning a cell replaces only its lab outputs.

[Lesson](../modules/09-run-the-application/lesson.md) · [Exercise](../modules/09-run-the-application/exercise.md)


## Prepare the notebook workspace

Run this cell after every kernel restart. It clears inherited `CWLTOOL_OPTIONS` so a Coder workspace configured for Podman can still run the explicitly local exercises. It does not change your cluster or publish anything.


In [ ]:
lab_start_dir="$PWD"
while [[ ! -f reference/waterbodies.cwl || ! -f pyproject.toml ]]; do
    if [[ "$PWD" == / ]]; then
        cd "$lab_start_dir"
        break
    fi
    cd ..
done
export WORKSPACE="$PWD"
if [[ -x "$WORKSPACE/.venv/bin/python" && -f "$WORKSPACE/reference/waterbodies.cwl" ]]; then
    source "$WORKSPACE/.venv/bin/activate"
    export PYTHONPATH="$WORKSPACE${PYTHONPATH:+:$PYTHONPATH}"
    export CWLTOOL_OPTIONS=
    export MPLBACKEND=Agg
    export LAB_DIR="$WORKSPACE/.notebook-runs/09-run-the-application"
    mkdir -p "$LAB_DIR"
    cd "$LAB_DIR"
    python --version
    waterbodies --help
else
    printf 'Open this notebook inside the course checkout. Ask the instructor to check workspace environment provisioning.\n' >&2
    false
fi


## Run the complete application locally

This path works inside the Kubernetes/Coder workspace without a nested container engine. The input Directory is staged next to the lab; the canonical workflow runs with `--no-container`.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
from pathlib import Path
from scripts.fixtures import create_fixture
create_fixture(Path('source'))
print('Created an 8 x 8 STAC scene with green and NIR assets under source/')
PYTHON

python - <<'PYTHON'
from pathlib import Path
import yaml
job = {'item': {'class': 'Directory', 'path': str(Path('source').resolve())},
       'aoi': '1,1,7,7', 'epsg': 'EPSG:4326', 'bands': ['green', 'nir']}
Path('inputs.yaml').write_text(yaml.safe_dump(job))
PYTHON
run_directory=$(mktemp -d "$LAB_DIR/workflow.XXXXXX")
cwltool --no-container --outdir "$run_directory" "$WORKSPACE/reference/waterbodies.cwl#waterbodies" inputs.yaml > results.json 2> runner.log
cat runner.log
cat results.json

python - <<'PYTHON'
from pathlib import Path
import numpy as np
import rasterio
from waterbodies.common import read_item
from waterbodies.validation import validate_item
import json
from urllib.parse import unquote, urlparse
product = json.loads(Path('results.json').read_text())['stac-catalog']
catalog_path = Path(unquote(urlparse(product['location']).path))
item = read_item(catalog_path)
validate_item(item)
href = item.assets['data'].get_absolute_href()
assert href is not None
with rasterio.open(href) as raster:
    pixels = raster.read(1)
    print('Grid:', raster.shape, 'Water pixels:', np.count_nonzero(pixels))
    assert raster.shape == (6, 6)
    assert np.count_nonzero(pixels) == 18
print('STAC validation passed:', item.id)
PYTHON
)


## Verify portability

Copy the complete catalog, including its raster asset. Resolve the relocated asset through PySTAC and validate the copied Item. Relative catalog links make the complete product portable.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
from pathlib import Path
import shutil
import numpy as np
import rasterio
from waterbodies.common import read_item
from waterbodies.validation import validate_item
import json
from urllib.parse import unquote, urlparse
product = json.loads(Path('results.json').read_text())['stac-catalog']
catalog_path = Path(unquote(urlparse(product['location']).path))
shutil.copytree(catalog_path, 'relocated/catalog', dirs_exist_ok=True)
item = read_item(Path('relocated/catalog'))
validate_item(item)
href = item.assets['data'].get_absolute_href()
assert href is not None
assert Path(href).is_relative_to(Path('relocated').resolve())
with rasterio.open(href) as raster:
    assert raster.shape == (6, 6)
    assert np.count_nonzero(raster.read(1)) == 18
print('Relocated product validates and resolves:', href)
PYTHON
python -m pytest "$WORKSPACE/tests/workflow"
)


## Optional: execute with Calrissian on Kubernetes

Calrissian creates tool pods instead of local processes. The instructor must supply a **digest-pinned published contract** (`LAB_K8S_CONTRACT`) and runnable job file (`LAB_K8S_INPUTS`), registry access and a shared writable `/calrissian` volume mounted at the same path in tool pods. Local `.notebook-runs/` files are not automatically shared with those pods. Use the platform's service account, volume configuration and node-selector file. Set the two paths and `LAB_RUN_KUBERNETES=1` in the next cell when that preparation is complete.

The cell records logs, result JSON and resource usage on the shared volume; the following cell inspects pods without opening a terminal.


In [ ]:
export LAB_RUN_KUBERNETES=${LAB_RUN_KUBERNETES:-0}
export LAB_K8S_CONTRACT=${LAB_K8S_CONTRACT:-}
export LAB_K8S_INPUTS=${LAB_K8S_INPUTS:-}


In [ ]:
(
set -euo pipefail
if [[ "$LAB_RUN_KUBERNETES" == 1 ]]; then
    : "${LAB_K8S_CONTRACT:?Set a prepared digest-pinned contract path}"
    : "${LAB_K8S_INPUTS:?Set the instructor-provided shared-volume job path}"
    test -r "$LAB_K8S_CONTRACT"
    test -r "$LAB_K8S_INPUTS"
    test -w /calrissian
    test -r /etc/calrissian/pod-node-selector.yaml
    python - <<'PYTHON'
import os
from pathlib import Path
from scripts.supply_chain import container_references
images = container_references(Path(os.environ['LAB_K8S_CONTRACT']))
assert all('@sha256:' in image for image in images), 'Use the promoted digest-pinned contract'
PYTHON
    run_directory=$(mktemp -d /calrissian/contract-lab.XXXXXX)
    calrissian --max-ram 3G --max-cores 2         --tmp-outdir-prefix "$run_directory/tmp/" --outdir "$run_directory/results"         --usage-report "$run_directory/usage.json" --tool-logs-basepath "$run_directory/logs"         --pod-nodeselectors /etc/calrissian/pod-node-selector.yaml         "$LAB_K8S_CONTRACT#waterbodies" "$LAB_K8S_INPUTS" > "$run_directory/results.json" 2> "$run_directory/runner.log"
    printf '%s\n' "$run_directory" > kubernetes-run.txt
    cat "$run_directory/results.json"
    cat "$run_directory/usage.json"
    python - <<'PYTHON'
import json
from pathlib import Path
from waterbodies.common import read_item
from waterbodies.validation import validate_item
run_directory = Path('kubernetes-run.txt').read_text().strip()
outputs = json.loads((Path(run_directory) / 'results.json').read_text())
product = outputs['stac-catalog']
location = product.get('path') or product['location']
from urllib.parse import unquote, urlparse
catalog_path = Path(unquote(urlparse(location).path))
item = read_item(catalog_path)
validate_item(item)
print('Kubernetes STAC validation passed:', item.id)
PYTHON
else
    printf 'Kubernetes execution skipped: configure the shared volume, published contract and staged job first.\n'
fi
)


In [ ]:
(
set -euo pipefail
if [[ "$LAB_RUN_KUBERNETES" == 1 ]]; then
    kubectl get pods
    run_directory=$(cat kubernetes-run.txt)
    cat "$run_directory/runner.log"
else
    printf 'Use the local runner log above; no cluster execution was requested.\n'
fi
)


## Record your findings

**Expected outcome:** The local and relocated catalog validate and contain an 18-pixel water mask. If configured, retain separate Calrissian logs, usage and validated result evidence.

Add your observations to a Markdown cell: what you predicted, what the command produced, and which contract declaration explains it. Restart the kernel and run all cells with the default settings to check reproducibility.

Compare [the solution](../modules/09-run-the-application/solution.md) after the activity. Return to [the lab index](README.md).
